# ITCS 6169/8169 — Computer Vision
## Assignment 1: The CNN Challenge
**Due: September 25, 2026 at 11:59 PM EDT**

---

In this assignment, you will build and train a convolutional neural network (CNN) for **16-class scene recognition** using a small dataset of **2,400 training images**.

This notebook is intentionally only a **starter baseline**. It provides a clean training pipeline and a deliberately simple CNN. Your task is to improve the system through informed decisions about architecture, optimization, augmentation, regularization, model selection, and training strategy.

### 2026 workflow: AI is your pair programmer
Use of an AI coding assistant (e.g., ChatGPT, Codex, GitHub Copilot, Claude Code, Gemini, Cursor) is **required**. You may use AI to help implement, debug, refactor, explain, or suggest experiments. However, **you are responsible for the scientific decisions and for verifying all generated code**.

Your final GitHub repository must include an `AI_USAGE.md` file describing representative AI-assisted tasks, at least one incorrect/ineffective AI suggestion, how you verified or corrected it, and one important decision that you made yourself.

### Important model restriction
The primary classifier for Assignment 1 must be a **CNN**. Do **not** use CLIP, DINO/DINOv2, VLM APIs, Vision Transformers as the primary model, or other foundation-model embeddings for your main submission. Pretrained CNNs are allowed if clearly documented.

### Google Colab
Google Colab is recommended, but not required. If using Colab, enable a GPU through **Runtime → Change runtime type → GPU**. You may also use the Educational Cluster or another resource for which you have authorized access.

In [ ]:
# Core imports
from pathlib import Path
import random
import time
import copy

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

import sys, json
import pandas as pd

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Optional: mount Google Drive when running in Colab.
# Skip this cell if you are running locally or on another server.
try:
    from google.colab import drive
    drive.mount('/content/gdrive')
except ImportError:
    print("Not running in Google Colab; Drive mount skipped.")

## 1. Set your working directory

Download the assignment dataset from the link provided in the assignment handout and organize it so that the notebook can find:

```text
data/
  train/
    class_1/
    ...
  test/
    class_1/
    ...
```

If you use Google Drive, change `PROJECT_ROOT` below to your assignment folder. **Do not hard-code a TA/instructor solution path.**

In [ ]:
# CHANGE THIS PATH to your own assignment directory if needed.
# Example for Colab + Google Drive:
# PROJECT_ROOT = Path('/content/gdrive/MyDrive/ITCS_6169_8169/assignment1')

# Locally, this notebook lives in `assignment1/` and the repository root (with data/, cvproj/,
# train.py, evaluate.py) is one level up, so we search upwards for the data folder.
PROJECT_ROOT = next(p for p in [Path('.').resolve(), *Path('.').resolve().parents] if (p / 'data' / 'train').exists())
DATA_ROOT = PROJECT_ROOT / 'data'
TRAIN_ROOT = DATA_ROOT / 'train'
TEST_ROOT = DATA_ROOT / 'test'
sys.path.insert(0, str(PROJECT_ROOT))  # makes the `cvproj` package importable

print('Project root:', PROJECT_ROOT.resolve())
print('Training data:', TRAIN_ROOT)
print('Testing data:', TEST_ROOT)

## 2. Reproducibility and device

A fixed seed makes the starter result easier to reproduce. Exact GPU reproducibility is not always guaranteed across hardware/software versions, so report your environment and seed in the final repository.

In [ ]:
SEED = 0

def set_random_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_random_seed(SEED)
from cvproj.utils import get_device
device = get_device()  # cuda > mps (Apple silicon) > cpu
print('Using device:', device)

## 3. Loading and preprocessing the data

The starter baseline converts images to grayscale and resizes them to $64\times64$. This is **not necessarily a good final choice**. You should decide whether color, resolution, normalization, and augmentation should be changed.

We split the provided training data into **training and validation sets**. Use the validation set for architecture/hyperparameter decisions. **Do not repeatedly tune on the test set.** The test set should be used for the final evaluation of a selected model.

In [ ]:
IMG_SIZE = 64
BATCH_SIZE = 64
VAL_FRACTION = 0.20

# Deliberately simple preprocessing for the starter baseline.
# TODO: improve this as part of your experiments.
baseline_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

full_train_dataset = datasets.ImageFolder(TRAIN_ROOT, transform=baseline_transform)
test_dataset = datasets.ImageFolder(TEST_ROOT, transform=baseline_transform)

class_names = full_train_dataset.classes
num_classes = len(class_names)
print(f'Classes ({num_classes}): {class_names}')
print(f'Total training images: {len(full_train_dataset)}')
print(f'Test images: {len(test_dataset)}')

val_size = int(round(len(full_train_dataset) * VAL_FRACTION))
train_size = len(full_train_dataset) - val_size
generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset = random_split(
    full_train_dataset, [train_size, val_size], generator=generator
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=0, pin_memory=torch.cuda.is_available())

print(f'Train: {len(train_dataset)} | Validation: {len(val_dataset)} | Test: {len(test_dataset)}')

In [ ]:
# Visualize a few training examples
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, image, label in zip(axes.flat, images[:8], labels[:8]):
    image = image * 0.5 + 0.5  # unnormalize
    ax.imshow(image.squeeze(0), cmap='gray')
    ax.set_title(class_names[label.item()])
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Training a CNN from scratch

Gone are the days when hand-designed features were the default solution. We will start with end-to-end learning and train a CNN directly for 16-way classification.

The architecture below is deliberately weak and simple. **Do not treat it as a recommended architecture.** It exists only to verify that the pipeline works and to give you a baseline to beat.

In [ ]:
class TNet(nn.Module):
    """A deliberately small CNN starter baseline."""
    def __init__(self, num_classes=16):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=4, stride=4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 15 * 15, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = TNet(num_classes=num_classes)
print(model)
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## 5. Training and evaluation utilities

The training loop below selects the checkpoint with the **best validation accuracy**. This is a much better experimental practice than evaluating the test set after every epoch.

You are welcome to rewrite this code. For example, you may add a scheduler, mixed precision, early stopping, checkpointing, experiment tracking, or richer metrics. If AI helps you implement such changes, verify the implementation and document representative usage in `AI_USAGE.md`.

In [ ]:
@torch.inference_mode()
def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    running_loss = 0.0
    criterion = nn.CrossEntropyLoss()

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = model(images)
        loss = criterion(logits, labels)
        running_loss += loss.item() * images.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


def train_model(model, train_loader, val_loader, optimizer, epochs=20, device=device):
    criterion = nn.CrossEntropyLoss()
    model = model.to(device)
    best_state = copy.deepcopy(model.state_dict())
    best_val_acc = 0.0
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    start = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_seen = 0

        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * images.size(0)
            total_seen += labels.size(0)

        train_loss = total_loss / total_seen
        val_loss, val_acc = evaluate(model, val_loader, device)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

        elapsed = time.time() - start
        print(f'Epoch {epoch:02d}/{epochs} | '
              f'train loss {train_loss:.4f} | val loss {val_loss:.4f} | '
              f'val acc {val_acc:.4f} | {elapsed:.1f}s')

    model.load_state_dict(best_state)
    print(f'Best validation accuracy: {best_val_acc:.4f}')
    return model, history

In [ ]:
# Train the deliberately simple baseline.
set_random_seed(SEED)
model = TNet(num_classes=num_classes)
optimizer = optim.Adam(model.parameters(), lr=0.002)

model, history = train_model(
    model, train_loader, val_loader, optimizer, epochs=20, device=device
)

In [ ]:
# Plot the starter training history.
plt.figure(figsize=(6, 4))
plt.plot(history['train_loss'], label='Train loss')
plt.plot(history['val_loss'], label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

plt.figure(figsize=(6, 4))
plt.plot(history['val_acc'], label='Validation accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

## 6. Your turn: beat the baseline

This simple model should only be treated as a sanity check. Your objective is to substantially improve it.

Rather than asking an AI tool to ``make the accuracy higher,'' formulate hypotheses yourself. Examples of useful questions include whether the model is overfitting, whether color matters, whether stronger augmentation helps, whether a different CNN family is more appropriate, whether pretrained CNN features transfer well, or whether optimization/regularization is limiting performance.

For each important experiment, write down:

| Experiment | What changed? | Why did you try it? | Validation accuracy | What did you learn? |
|---|---|---|---:|---|
| Baseline | Starter TNet | Sanity check | -- | -- |
| Experiment 1 | | | | |
| Experiment 2 | | | | |
| Final model | | | | |

**Do not use the test set to choose among these experiments.**

### 6.1 Experimental protocol

The notebook above reproduces the starter baseline. All further experiments are run with the scripts in the repository root so that every number below can be reproduced from a config file:

```bash
python train.py --config configs/<experiment>.yaml      # trains, keeps the best-val-accuracy epoch
python evaluate.py --checkpoint checkpoints/<run>.pt --tta   # val + test accuracy, per-class, confusion matrix
```

**Rules I followed so that comparisons are meaningful**

* **One fixed validation split.** A *stratified* 80/20 split of `data/train` (30 images per class held out → 1,920 train / 480 val), seed 0, identical for every run (`cvproj/data.py::stratified_split`). The starter used a non-stratified `random_split`; with only 150 images per class, stratifying removes class-balance noise from the val estimate.
* **The test set (400 images, 25 per class) is never used for any decision.** It is evaluated once, for the selected final model, at the end of this notebook.
* **One change at a time** between consecutive experiments (each config's first line says what changed and why).
* **Model selection** = epoch with the highest validation accuracy (ties → lower val loss). I also log the *last-epoch* val accuracy; a large gap between best and last epoch would indicate the selection itself is overfitting the 480-image val set.
* **Noise floor.** One val image = 0.21 %. Differences below ~1 % (≈5 images) are within seed-to-seed noise, so I re-run important comparisons with extra seeds before drawing conclusions.
* Hardware: Apple M4 (PyTorch MPS backend). A throughput check (`scripts/bench.py`) showed that `channels_last` memory format is 2–3× *slower* on MPS, so it is not used.

Set `RUN_TRAINING = True` below to re-train every experiment from inside the notebook (≈2–3 h on an M4); by default the cells load the logged results in `results/`.

In [ ]:
import subprocess

RUN_TRAINING = False   # True = re-run train.py for each experiment (slow); False = load logged results
RESULTS = PROJECT_ROOT / 'results'

def run_experiment(name):
    if RUN_TRAINING:
        subprocess.run([sys.executable, 'train.py', '--config', f'configs/{name}.yaml'],
                       cwd=PROJECT_ROOT, check=True)
    return json.load(open(RESULTS / 'runs' / f'{name}.json'))

def summary(names):
    rows = []
    for n in names:
        r = json.load(open(RESULTS / 'runs' / f'{n}.json'))
        h = r['history']
        rows.append({'experiment': n, 'params (M)': round(r['params'] / 1e6, 2),
                     'img': r['config']['data']['img_size'], 'aug': r['config']['data'].get('augment'),
                     'best val acc': round(r['best']['val_acc'], 4), 'best epoch': r['best']['epoch'],
                     'last-epoch val acc': round(r['final_epoch']['val_acc'], 4),
                     'val acc (flip TTA)': round(r['val_acc_tta'], 4),
                     'final train acc': round(h['train_acc'][-1], 4),
                     # median epoch time x epochs: robust to the laptop sleeping mid-run (happened once, in e4)
                     'train time (min)': round(np.median(np.diff([0] + h['time'])) * len(h['time']) / 60, 1)})
    return pd.DataFrame(rows).set_index('experiment')

def plot_runs(names, title=''):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for n in names:
        h = json.load(open(RESULTS / 'runs' / f'{n}.json'))['history']
        ep = range(1, len(h['val_acc']) + 1)
        line, = axes[0].plot(ep, h['train_loss'], '--', alpha=0.6)
        axes[0].plot(ep, h['val_loss'], color=line.get_color(), label=n)
        axes[1].plot(ep, h['train_acc'], '--', color=line.get_color(), alpha=0.6)
        axes[1].plot(ep, h['val_acc'], color=line.get_color(), label=n)
    axes[0].set(xlabel='epoch', ylabel='loss (dashed = train)', title=f'{title} loss')
    axes[1].set(xlabel='epoch', ylabel='accuracy (dashed = train)', title=f'{title} accuracy', ylim=(0, 1.01))
    axes[1].legend(fontsize=8)
    plt.tight_layout(); plt.show()

### 6.2 Phase 1 — a better CNN trained from scratch

**Diagnosis of the starter baseline (e0).** Train accuracy reaches 99 % while validation accuracy peaks at 44 % and validation loss rises from epoch ~5 onwards: the model is *severely overfitting*, and it is also a poor feature extractor (one conv layer, 4×4 max-pool, then a 3,600→16 linear layer that holds 93 % of its parameters).

**e1 — `SceneCNN` (`cvproj/models.py`).** Hypothesis: the baseline lacks *depth/receptive field*, not parameters. A VGG-style network with four stages of two 3×3 conv + BatchNorm + ReLU, 2×2 max-pool, then global-average-pooling + dropout 0.3 + linear (1.18 M params, almost all convolutional). Input 128×128 (3-channel replicated grey so the same pipeline serves ImageNet models later), AdamW (lr 1e-3, wd 0.05), cosine schedule with 2 warm-up epochs, 60 epochs, batch 32.

**e2 — + strong augmentation.** Hypothesis: e1 still overfits (100 % train), so label-preserving augmentation (RandomResizedCrop scale 0.35–1, horizontal flip, brightness/contrast jitter, ±10° rotation) should close the train/val gap.

In [ ]:
phase1 = ['e0_baseline_tnet', 'e1_scenecnn', 'e2_scenecnn_aug']
for n in phase1: run_experiment(n)
display(summary(phase1))
plot_runs(phase1, 'Phase 1:')

### 6.3 Phase 2 — does ImageNet initialization help, and how should we fine-tune?

**e3 vs e4 (controlled).** Same ResNet-18, same 224×224 input, same strong augmentation, same AdamW + cosine schedule. e3 starts from random weights (60 epochs, lr 1e-3); e4 starts from torchvision's `ResNet18_Weights.IMAGENET1K_V1` and fine-tunes **all** layers with a new 16-way linear head (30 epochs, lr 3e-4 — a smaller LR so the pretrained filters are adapted rather than overwritten).

**e7 — linear probe.** Same as e5 but every pretrained layer is frozen and only the linear head is trained. This measures how much of e4/e5's gain comes from the ImageNet features *as they are* versus from adapting them.

**e5 — + label smoothing 0.1.** Hypothesis: e4 reaches 99 % train accuracy with confident predictions; with only 120 images per class and several visually overlapping classes (e.g. *Bedroom/LivingRoom*, *Industrial/Store*, *OpenCountry/Coast*), softening the targets should reduce over-confidence and improve calibration/generalization.

**e6 — + MixUp/CutMix** (50 % of batches, MixUp α=0.2 or CutMix α=1.0). Hypothesis: a stronger regularizer than label smoothing for the remaining overfitting.

In [ ]:
phase2 = ['e3_resnet18_scratch', 'e4_resnet18_pretrained', 'e7_r18_linearprobe', 'e5_r18_labelsmooth', 'e6_r18_mixcut']
for n in phase2: run_experiment(n)
display(summary(phase2))
plot_runs(['e3_resnet18_scratch', 'e4_resnet18_pretrained', 'e7_r18_linearprobe'], 'Scratch vs ImageNet init:')
plot_runs(['e4_resnet18_pretrained', 'e5_r18_labelsmooth', 'e6_r18_mixcut'], 'Training strategy:')

### 6.4 Phase 3 — does the CNN family / capacity matter once we start from ImageNet?

Every model below uses **exactly the e5 recipe** (224×224, strong augmentation, AdamW lr 3e-4, wd 0.05, cosine with 1 warm-up epoch, label smoothing 0.1, 30 epochs, batch 32) and differs only in the architecture, so differences are attributable to the backbone. GFLOPs are forward-pass multiply-adds ×2 for one 224×224 image (counted with `torch.utils.flop_counter`); train time is on the M4 GPU.

In [ ]:
from torch.utils.flop_counter import FlopCounterMode
from cvproj.models import build_model

def gflops(model_cfg, img_size):
    m = build_model(dict(model_cfg, pretrained=False), 16).eval()
    with FlopCounterMode(display=False) as fc:
        m(torch.randn(1, 3, img_size, img_size))
    return fc.get_total_flops() / 1e9

phase3 = ['e5_r18_labelsmooth', 'e10_resnet50', 'e11_mobilenet_v3', 'e9_efficientnet_b0', 'e8_convnext_tiny']
for n in phase3: run_experiment(n)
arch = summary(phase3)
arch.insert(1, 'GFLOPs', [round(gflops(json.load(open(RESULTS / 'runs' / f'{n}.json'))['config']['model'], 224), 2) for n in phase3])
display(arch)

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(arch['train time (min)'], arch['best val acc'] * 100, s=arch['params (M)'] * 15 + 20, alpha=0.6)
for n, r in arch.iterrows():
    ax.annotate(n.split('_', 1)[1], (r['train time (min)'], r['best val acc'] * 100), fontsize=8, xytext=(4, 4), textcoords='offset points')
ax.set(xlabel='training time on M4 (min, 30 epochs)', ylabel='best val accuracy (%)', title='Accuracy vs cost (marker area ~ params)')
plt.show()

#### Is the label-smoothing gain real? (seed check)

e4 → e5 is a 2.3 % jump on one seed, i.e. 11 validation images — close enough to the noise floor that I re-trained both configs with two more training seeds (same split; the seed changes the head initialisation, data order and augmentations).

In [ ]:
def seed_table(bases, seeds=(0, 1, 2)):
    rows = []
    for b in bases:
        accs = []
        for s in seeds:
            f = RESULTS / 'runs' / (f'{b}.json' if s == 0 else f'{b}_seed{s}.json')
            if f.exists():
                accs.append(json.load(open(f))['best']['val_acc'] * 100)
        rows.append({'config': b, 'n seeds': len(accs), 'val acc per seed (%)': ', '.join(f'{a:.2f}' for a in accs),
                     'mean (%)': round(np.mean(accs), 2), 'std (%)': round(np.std(accs, ddof=1), 2) if len(accs) > 1 else None})
    return pd.DataFrame(rows).set_index('config')

seed_table(['e4_resnet18_pretrained', 'e5_r18_labelsmooth'])

### 6.5 Phase 4 — does input resolution justify its cost?

Same e5 recipe (ResNet-18), only `img_size` changes. The source images are 256×256, so 288 is a mild *up*-sampling: it adds no new pixels but makes every object larger relative to the fixed receptive fields / stride-32 output grid of an ImageNet-trained network (ImageNet objects are typically larger in the frame than in these wide scene shots). Cost grows ~quadratically with the side length.

In [ ]:
phase4 = ['e12_r18_res160', 'e5_r18_labelsmooth', 'e13_r18_res288']
for n in phase4: run_experiment(n)
res = summary(phase4)
res.insert(1, 'GFLOPs', [round(gflops({'name': 'resnet18'}, s), 2) for s in res['img']])
display(res)

# Does the resolution gain transfer to other backbones? (224 = mean over seeds 0/1, 288 = seed 0)
pairs = [('resnet18', 'e5_r18_labelsmooth', 'e13_r18_res288'), ('resnet50', 'e10_resnet50', 'e14_r50_res288'),
         ('efficientnet_b0', 'e9_efficientnet_b0', 'e15_effb0_res288')]
rows = []
for arch_name, a, b in pairs:
    run_experiment(b)
    st = seed_table([a], seeds=(0, 1))
    rows.append({'model': arch_name, 'val acc @224 (seed mean)': st['mean (%)'].iloc[0],
                 'val acc @288': round(json.load(open(RESULTS / 'runs' / f'{b}.json'))['best']['val_acc'] * 100, 2),
                 'GFLOPs @224': round(gflops({'name': arch_name}, 224), 2), 'GFLOPs @288': round(gflops({'name': arch_name}, 288), 2)})
pd.DataFrame(rows).set_index('model')

#### Follow-up: is ConvNeXt limited by the shared learning rate?

In e8 ConvNeXt-Tiny's validation accuracy oscillated between 91 % and 94 % for most of training, and only reached 95.2 % once the cosine schedule had decayed the LR. That pattern suggests the step size (3e-4, tuned on ResNet-18) is too large for this backbone, which was itself pre-trained with AdamW and is known to prefer smaller fine-tuning LRs. e8b changes only the LR to 1e-4.

In [ ]:
for n in ['e8_convnext_tiny', 'e8b_convnext_tiny_lr1e-4']: run_experiment(n)
display(summary(['e8_convnext_tiny', 'e8b_convnext_tiny_lr1e-4']))
plot_runs(['e8_convnext_tiny', 'e8b_convnext_tiny_lr1e-4'], 'ConvNeXt-T LR:')

**What happened in phase 1**

* **e0 → e1 (+43.7 %).** Depth, BatchNorm and global average pooling turn a 44 % model into an 87.7 % one with only 1.2 M parameters. e1 still reaches 100 % train accuracy, but its val loss stays low: the GAP head generalises far better than the starter's 3,600→16 linear layer.
* **e1 → e2: strong augmentation did *not* help (87.7 → 87.3 %).** Train accuracy dropped from 100 % to 90 % and the best epoch was the very last one, so my first reading was *under*-fitting within the 60-epoch budget. That hypothesis is tested in the failure analysis below (§6.6).

### 6.6 Failure analysis — why didn't strong augmentation help?

**What I tried.** The "strong" pipeline: `RandomResizedCrop(scale=0.35–1)`, horizontal flip, brightness/contrast jitter (0.4) and ±10° rotation (e2), the standard recipe for ImageNet-style training.

**Why I expected it to work.** e1 overfits (100 % train accuracy on 1,920 images), and augmentation is the usual remedy when labelled data are scarce.

**What happened.** Val accuracy went *down* slightly (87.7 → 87.3 %). I tested three explanations, one change at a time:

| run | change vs e2 | hypothesis tested | best val acc |
|---|---|---|---:|
| e1 | no augmentation | reference | 87.7 % |
| e2 | strong aug, 60 ep | — | 87.3 % |
| e2c | strong aug, **150 epochs** | "it just needs longer to converge" | 88.1 % |
| e2d | strong aug, crop scale **0.8–1** | "aggressive crops destroy the scene layout" | 88.8 % |
| e2b | **light** aug (pad-crop + flip only) | "only label-preserving geometry is useful" | **90.2 %** |
| e5 / e5b | pretrained ResNet-18, strong vs **light** | does this hold for fine-tuning too? | 94.4 % (3-seed mean) vs 95.4 % |

**What I learned.**
1. *Longer training is not the fix* (e2c): with 2.5× the compute, train accuracy rises to 97 % but val accuracy plateaus at ~88 %. So the problem is not under-fitting; the augmented training distribution is simply less useful for the test distribution.
2. *Scene categories are defined by global layout* (horizon position, sky/ground ratio, the vertical structure of a tall building). A 35 %-area crop of a *Coast* image can look like *OpenCountry*, and a crop of a *Kitchen* like an *Office*. Restricting the crop scale (e2d) recovers part of the loss, and the remaining components (rotation, which also fills the corners with black triangles that never occur at test time, and photometric jitter on greyscale images) still cost accuracy relative to light augmentation.
3. *The lesson transfers to fine-tuning* (e5b): the pretrained ResNet-18 is also better with light augmentation. Augmentation must be **label-preserving for the specific task**. Object-centric ImageNet defaults are not automatically right for scene recognition.

(Each of these is a single-seed comparison; one val image is 0.21 %. The e2b-vs-e2 gap of 2.9 % ≈ 14 images is well above the seed-to-seed spread we measured (≈0.4–0.6 %), and the e5b gap is re-checked with a second seed below.)

In [ ]:
aug_runs = ['e1_scenecnn', 'e2_scenecnn_aug', 'e2c_scenecnn_aug_long', 'e2d_scenecnn_aug_mildcrop', 'e2b_scenecnn_lightaug']
for n in aug_runs + ['e5b_r18_lightaug']: run_experiment(n)
display(summary(aug_runs))
plot_runs(['e1_scenecnn', 'e2_scenecnn_aug', 'e2c_scenecnn_aug_long', 'e2b_scenecnn_lightaug'], 'Augmentation (SceneCNN):')
seed_table(['e5_r18_labelsmooth', 'e5b_r18_lightaug'])

In [ ]:
# What the two augmentation pipelines actually feed the network (same source image, 6 samples each)
from cvproj.data import build_transforms, IMAGENET_MEAN, IMAGENET_STD
from PIL import Image
src = Image.open(sorted((TRAIN_ROOT / 'Coast').glob('*.jpg'))[0]).convert('L')
fig, axes = plt.subplots(2, 7, figsize=(14, 4.4))
for row, aug in enumerate(['light', 'strong']):
    tf = build_transforms({'img_size': 224, 'augment': aug}, train=True)
    axes[row, 0].imshow(src.resize((224, 224)), cmap='gray'); axes[row, 0].set_title('original' if row == 0 else '', fontsize=9)
    for k in range(1, 7):
        x = tf(src)[0] * IMAGENET_STD[0] + IMAGENET_MEAN[0]
        axes[row, k].imshow(x.clamp(0, 1), cmap='gray', vmin=0, vmax=1)
    axes[row, 1].set_title(f'{aug} augmentation samples →', fontsize=9, loc='left')
for ax in axes.flat: ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# FINAL TEST EVALUATION
# Run this only after you have selected your final model using validation results.
# Replace `model` with your final selected model/checkpoint.

test_loss, test_acc = evaluate(model, test_loader, device)
print(f'Final test loss: {test_loss:.4f}')
print(f'Final test accuracy: {test_acc:.4f}')

## 7. AI-assisted development checkpoint

Before submission, create `AI_USAGE.md` in your GitHub repository. You do **not** need to provide every prompt. Include:

1. the AI coding tool(s) you used;
2. 3--5 representative ways AI assisted you;
3. at least one AI suggestion that was incorrect, ineffective, or questionable;
4. how you tested, verified, corrected, or rejected that suggestion; and
5. one important architecture/experimental decision that **you** made.

A good entry describes the engineering/research interaction rather than saying only, ``I used ChatGPT to write code.''

### A useful habit
When AI generates code, ask yourself before running it:
- What do I expect this code to do?
- What assumption is it making about tensor shape/data/model behavior?
- How will I know if it is wrong?
- What experiment would distinguish a real improvement from noise?

## 8. GitHub and submission requirements

All assignment code must be maintained in **GitHub** (not GitLab). Your repository should contain a meaningful commit history and, at minimum:

```text
README.md
AI_USAGE.md
training / model code
inference or evaluation code
requirements.txt or environment information
final checkpoint or a link to it
reproduction instructions
```

Do not create the entire repository as a single commit immediately before the deadline. Use Git throughout development as you would in a real research or engineering project.

Your final submission is a **maximum two-page PDF report** containing the working GitHub link. The report should clearly state your best result, validation/model-selection strategy, final recipe, important experiments, at least one failure analysis, and a brief AI+human reflection.

Friendly discussion and brainstorming are encouraged, but your implementation, experiments, report, and repository must be your own. Do not share code, checkpoints, detailed hyperparameters, or test predictions before the deadline.

The instructor may ask you to explain any important component of your submitted system. You are responsible for understanding all code in your final repository, including AI-generated code.

**Deadline: September 25, 2026 at 11:59 PM EDT.**

Good luck — and treat this as your first small computer vision research project of the semester.